# Chapter-8：旅行多智能体系统（LangGraph 固定图 + A2A + MCP）

> Last updated: 2026-11-22

## 学习目标

第 6 章虽在中心编排器内完成了任务拆解、依赖分析与领域智能体路由，但智能体与编排器运行于同一进程，工具仍停留在第 5 章引入的本地检索函数层面，天气与航班仅返回占位结果；第 7 章分别演示了 MCP 工具接入与 A2A 智能体发布，却未将二者同中心编排贯通。本章在此之上实现了一套完整的智能体编排：以 LangGraph 固定图作为中心编排，把 5 个领域智能体封装为 A2A 服务并经 Agent Card 注册，同时将天气、地图、酒店、餐厅、航班统一交由第三方 MCP 提供。这些智能体各司其职、按依赖分层并行执行，再汇总成统一答复，以合作的形式共同解决复杂的旅行规划问题。重点理解三件事：

1. 完整的智能体编排：中心编排采用 LangGraph 固定图，`pre_survey → retrieve_memory → build_plan → route → execute_layer → aggregate → save_memory` 依次对应第 2、3、4、6、5+ 章，各智能体按依赖协同完成复杂问题。
2. 子智能体先以 A2A 服务启动，通过 Agent Card 注册到中心智能体，形成可独立部署、跨语言调用的服务边界。
3. 天气、地图/酒店/餐厅、航班等能力统一交由第三方 MCP 提供，并在子智能体服务生命周期内保持常驻会话。


## 1. 环境准备与模块导入

使用书仓库统一 `.venv`，并从仓库根目录 `.env` 加载配置。

In [12]:
import asyncio
import os
import subprocess
import sys
import time
from pathlib import Path

_CHAPTER8_DIR = Path.cwd()
if not (_CHAPTER8_DIR / "travel_multi_agent").exists():
    _CHAPTER8_DIR = Path.cwd() / "Chapter-8"
if not (_CHAPTER8_DIR / "travel_multi_agent").exists():
    raise FileNotFoundError("请在 Chapter-8 目录或书仓库根目录运行本 Notebook")

_BOOK_ROOT = _CHAPTER8_DIR.parent
for _p in (_CHAPTER8_DIR, _BOOK_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import numpy  # 预加载 numpy DLL，避免 Windows 下延迟加载偶发失败

from travel_multi_agent.config import AGENT_CARDS_DIR, load_project_dotenv
from travel_multi_agent.agents.registry import DomainAgentName, create_travel_registry
from travel_multi_agent import mcp_tools
from travel_multi_agent.a2a import settings as a2a_settings
from travel_multi_agent.a2a.client import A2ARegistry, discover_agent_card
from llm import model_configured

load_project_dotenv()

# 演示时降低可观测性噪音：关闭 span 控制台输出、提高日志级别。
# 若要看完整链路，可删除下面两行，或将 OTEL_TRACES_EXPORTER 设为 console。
os.environ["OTEL_TRACES_EXPORTER"] = "none"
os.environ["LOG_LEVEL"] = "WARNING"

print("环境就绪。模型已配置：", model_configured())

环境就绪。模型已配置： True


## 2. MCP 工具注册（常驻会话）

所有 MCP 在 `mcp_tools.MCP_SERVERS` 里一次性注册。`load_tools()` 打开一个**常驻会话**并复用，长驻的 A2A 服务不会在每次工具调用时都重启 `npx` 进程。

In [13]:
print("本章注册的 MCP 服务：", list(mcp_tools.MCP_SERVERS))


本章注册的 MCP 服务： ['weather', 'baidu', 'variflight']


## 3. 领域智能体目录（A2A Agent 工厂）

`registry.py` 是唯一事实源：`DomainAgentName` 枚举 + 能力目录（供路由/拆解复用第 6 章），以及供 A2A 服务侧构建 agent 的工厂。

In [14]:
registry = create_travel_registry()
print("=== 领域智能体能力目录 ===\n")
for name in registry.get_agent_names():
    info = registry.agents[name]
    print(f"- {name}: {info['description']}")
    for skill in info["skills"]:
        print(f"    {skill['name']}  inputSchema={skill['inputSchema']}  outputSchema={skill['outputSchema']}")

=== 领域智能体能力目录 ===

- WeatherAgent: 查询指定城市、日期的天气预报，提供温度、天气状况和出行建议
    weather_query  inputSchema=['city', 'date']  outputSchema=['forecast']
- HotelAgent: 根据位置、预算、偏好（近景区/安静/品牌）推荐酒店；地图关键词与主观偏好分离
    hotel_search  inputSchema=['city', 'preferences', 'budget']  outputSchema=['hotels']
- RestaurantAgent: 根据菜系、位置、预算推荐当地特色餐厅和美食
    restaurant_search  inputSchema=['location', 'cuisine', 'budget']  outputSchema=['restaurants']
- ItineraryAgent: 综合旅行出发时间和返回时间、天气、景点、酒店、餐厅和航班结果，生成最终旅行日程与用户答复。
    itinerary_synthesis  inputSchema=['cities', 'dates', 'preferences', 'departure_city', 'destination_city', 'days']  outputSchema=['final_response']
- FlightAgent: 查询出发地到目的地的航班信息、价格和时刻表
    flight_search  inputSchema=['departure', 'arrival', 'date']  outputSchema=['flights']


## 4. A2A Agent Card 注册

中心智能体从 `agents/agent_cards/*.json` 读取 Agent Card（名字 → endpoint），执行层据此调用服务。

In [15]:
a2a_registry = A2ARegistry.from_directory(AGENT_CARDS_DIR)
print("=== A2A Agent Card 注册 ===\n")
for name in a2a_registry.names():
    print(f"- {name}  ->  {a2a_registry.endpoint_for(name)}")

=== A2A Agent Card 注册 ===

- FlightAgent  ->  http://127.0.0.1:9034/
- HotelAgent  ->  http://127.0.0.1:9032/
- ItineraryAgent  ->  http://127.0.0.1:9035/
- RestaurantAgent  ->  http://127.0.0.1:9033/
- WeatherAgent  ->  http://127.0.0.1:9031/


## 5. 启动 5 个 A2A 子智能体服务

每个领域智能体一个进程（端口见 `a2a/settings.py`）。启动后轮询 `/.well-known/agent-card.json` 直到就绪。若本机未装 Node.js 或缺少 MCP Key，服务会启动失败，按需逐项排查。

In [16]:
SERVICES_READY = model_configured()
_started: list = []

async def _wait_ready(url: str, timeout: float = 90.0) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            await discover_agent_card(url)
            return
        except Exception:
            await asyncio.sleep(1.0)
    raise TimeoutError(f"A2A 服务未就绪: {url}")

async def _start_services() -> None:
    env = dict(os.environ)
    env["PYTHONPATH"] = os.pathsep.join([str(_CHAPTER8_DIR), str(_BOOK_ROOT)])
    for member in DomainAgentName:
        port = a2a_settings.service_port(member)
        url = a2a_settings.service_base_url(member)
        card = AGENT_CARDS_DIR / f"{member.name.lower()}_agent.json"
        cmd = [
            sys.executable, "-m", "travel_multi_agent.a2a.server",
            "--agent-card", str(card),
            "--host", a2a_settings.server_host(),
            "--port", str(port),
            "--public-url", url,
        ]
        proc = subprocess.Popen(
            cmd, cwd=str(_CHAPTER8_DIR), env=env,
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        _started.append(proc)
        await _wait_ready(url)
        print(f"  ✓ {member.value} @ {url}")

if SERVICES_READY:
    await _start_services()
    print(f"\n已启动 {len(_started)} 个 A2A 服务。")
else:
    print("模型未配置，跳过 A2A 服务启动；请配置 API Key 后重跑本 Cell。")

  ✓ WeatherAgent @ http://127.0.0.1:9031/
  ✓ HotelAgent @ http://127.0.0.1:9032/
  ✓ RestaurantAgent @ http://127.0.0.1:9033/
  ✓ FlightAgent @ http://127.0.0.1:9034/
  ✓ ItineraryAgent @ http://127.0.0.1:9035/

已启动 5 个 A2A 服务。


## 6. 运行中心编排器（固定图）

`LangGraphOrchestrator` 编译出固定图，执行层通过 A2A 调用上面启动的子智能体；聚合用代码选择最终回复，不再走聚合 LLM。

In [17]:
from travel_multi_agent.orchestration.fixed_graph import LangGraphOrchestrator

QUERY = "帮我规划一个上海三天两夜的旅行，喜欢安静的酒店，预算每晚不超过 800 元。"

if SERVICES_READY and _started:
    orchestrator = LangGraphOrchestrator(enable_memory=False)
    result = await orchestrator.process_request(QUERY, thread_id="notebook_demo")
    print("=== 最终回复 ===\n")
    print(result["final_response"])
    print("\n=== 子任务结果 ===")
    for task_id, item in (result.get("subtask_results") or {}).items():
        print(f"  {task_id} -> {item.get('agent')} [{item.get('status')}]")
else:
    print("A2A 服务未启动，跳过中心编排演示；请先运行上一步 Cell。")

time=2026-11-22T16:07:49.029376+00:00 level=INFO logger=travel_multi_agent.tracing.trace_provider trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=01d4343da91f178d msg=span.start span=latc.travel-multi-agent.request thread_id=notebook_demo user_query=帮我规划一个上海三天两夜的旅行，喜欢安静的酒店，预算每晚不超过 800 元。
time=2026-11-22T16:07:49.031390+00:00 level=INFO logger=travel_multi_agent.orchestration.fixed_graph.orchestrator trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=01d4343da91f178d msg=request.start query_preview=帮我规划一个上海三天两夜的旅行，喜欢安静的酒店，预算每晚不超过 800 元。 thread_id=notebook_demo
time=2026-11-22T16:07:49.035411+00:00 level=INFO logger=travel_multi_agent.tracing.trace_provider trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=c78cc728c1412a43 msg=span.start span=latc.travel-multi-agent.orchestration.pre_survey state={'thread_id': 'notebook_demo', 'user_query': '帮我规划一个上海三天两夜的旅行，喜欢安静的酒店，预算每晚不超过 800 元。', 'enable_memory': False, 'enable_s
time=2026-11-22T16:07:49.036543+00:00 level=INFO logger=travel_multi_age

{
    "name": "latc.travel-multi-agent.planner.pre_survey",
    "context": {
        "trace_id": "0x7ddc2857389ab2f4de296778d922b9a1",
        "span_id": "0x3dd3ca330fcfbc5a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc78cc728c1412a43",
    "start_time": "2026-11-22T16:07:49.038248Z",
    "end_time": "2026-11-22T16:07:56.271255Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "user.query": "\u5e2e\u6211\u89c4\u5212\u4e00\u4e2a\u4e0a\u6d77\u4e09\u5929\u4e24\u591c\u7684\u65c5\u884c\uff0c\u559c\u6b22\u5b89\u9759\u7684\u9152\u5e97\uff0c\u9884\u7b97\u6bcf\u665a\u4e0d\u8d85\u8fc7 800 \u5143\u3002"
    },
    "events": [
        {
            "name": "request",
            "timestamp": "2026-11-22T16:07:49.038248Z",
            "attributes": {
                "user_query": "\u5e2e\u6211\u89c4\u5212\u4e00\u4e2a\u4e0a\u6d77\u4e09\u5929\u4e24\u591c\u7684\u65c5\u884c\uff0c\u559c\u6b22\u5b89\u9759\u7684\u9152\u5e97\uff0

time=2026-11-22T16:08:00.379856+00:00 level=INFO logger=httpx2 trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=3cb9ece618f77706 msg=HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
time=2026-11-22T16:08:03.322213+00:00 level=INFO logger=httpx2 trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=3cb9ece618f77706 msg=HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
time=2026-11-22T16:08:03.325326+00:00 level=INFO logger=travel_multi_agent.tracing.trace_provider trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=3cb9ece618f77706 msg=span.ok span=latc.travel-multi-agent.planner.build_plan
time=2026-11-22T16:08:03.325326+00:00 level=INFO logger=travel_multi_agent.orchestration.fixed_graph.nodes trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=6f394655db01acde msg=✓ 共 4 个子任务，2 个执行层 thread_id=notebook_demo
time=2026-11-22T16:08:03.327812+00:00 level=INFO logger=travel_multi_agent.orch

{
    "name": "latc.travel-multi-agent.planner.build_plan",
    "context": {
        "trace_id": "0x7ddc2857389ab2f4de296778d922b9a1",
        "span_id": "0x3cb9ece618f77706",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6f394655db01acde",
    "start_time": "2026-11-22T16:07:56.281571Z",
    "end_time": "2026-11-22T16:08:03.325326Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "user.query": "\u5e2e\u6211\u89c4\u5212\u4e00\u4e2a\u4e0a\u6d77\u4e09\u5929\u4e24\u591c\u7684\u65c5\u884c\uff0c\u559c\u6b22\u5b89\u9759\u7684\u9152\u5e97\uff0c\u9884\u7b97\u6bcf\u665a\u4e0d\u8d85\u8fc7 800 \u5143\u3002"
    },
    "events": [
        {
            "name": "request",
            "timestamp": "2026-11-22T16:07:56.281571Z",
            "attributes": {
                "user_query": "\u5e2e\u6211\u89c4\u5212\u4e00\u4e2a\u4e0a\u6d77\u4e09\u5929\u4e24\u591c\u7684\u65c5\u884c\uff0c\u559c\u6b22\u5b89\u9759\u7684\u9152\u5e97\uff0

time=2026-11-22T16:08:09.391302+00:00 level=INFO logger=httpx2 trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=803f69b6ec9f4f94 msg=HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
time=2026-11-22T16:08:09.395734+00:00 level=INFO logger=travel_multi_agent.orchestration.fixed_graph.nodes trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=803f69b6ec9f4f94 msg=  路由结果: T1:HotelAgent → T2:RestaurantAgent → T3:WeatherAgent → T4:ItineraryAgent thread_id=notebook_demo
time=2026-11-22T16:08:09.397749+00:00 level=INFO logger=travel_multi_agent.orchestration.fixed_graph.nodes trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=803f69b6ec9f4f94 msg=plan.routed agent_count=4 routing_error_count=0 thread_id=notebook_demo
time=2026-11-22T16:08:09.397749+00:00 level=INFO logger=travel_multi_agent.tracing.trace_provider trace_id=7ddc2857389ab2f4de296778d922b9a1 span_id=803f69b6ec9f4f94 msg=span.ok span=latc.travel-multi-agent.orchestration.route
tim

{
    "name": "latc.travel-multi-agent.orchestration.route",
    "context": {
        "trace_id": "0x7ddc2857389ab2f4de296778d922b9a1",
        "span_id": "0x803f69b6ec9f4f94",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x01d4343da91f178d",
    "start_time": "2026-11-22T16:08:03.331189Z",
    "end_time": "2026-11-22T16:08:09.397749Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "state.thread_id": "notebook_demo",
        "state.user_query": "\u5e2e\u6211\u89c4\u5212\u4e00\u4e2a\u4e0a\u6d77\u4e09\u5929\u4e24\u591c\u7684\u65c5\u884c\uff0c\u559c\u6b22\u5b89\u9759\u7684\u9152\u5e97\uff0c\u9884\u7b97\u6bcf\u665a\u4e0d\u8d85\u8fc7 800 \u5143\u3002",
        "state.enable_memory": false,
        "state.enable_stream": false,
        "state.current_layer_index": 0
    },
    "events": [
        {
            "name": "request",
            "timestamp": "2026-11-22T16:08:03.331189Z",
            "attributes": {
       

... (4 outputs left out)


[多智能体自然语言协作] T1
发送方：HotelAgent
接收方：Supervisor
消息内容：
现在我有足够的酒店信息来进行推荐。根据预算（每晚不超过800元）和已有的酒店信息，我选择评分较高的几家酒店进行推荐。

需要注意的是：
1. 大部分酒店没有明确的价格字段，但经济型酒店和三星级酒店通常价格在800元以内
2. "安静"是主观属性，工具无法提供验证，我不能声称已核验
3. 我需要基于区域、地址、参考价格、评分和类型来推荐

让我整理推荐结果：

**推荐酒店（基于上海地区，预算每晚不超过800元）：**

1. **汉庭酒店(上海青浦新城地铁站店)**
   - 地址：华科路109号（青浦区）
   - 参考价格：未显示（经济型酒店通常在预算范围内）
   - 评分：4.8
   - 推荐理由：评分高达4.8分，位于青浦新城地铁站附近，交通便利，经济型酒店性价比高。

2. **上海南泉酒店(陆家嘴塘桥地铁站店)**
   - 地址：南泉路899号（浦东新区）
   - 参考价格：297元/晚
   - 评分：4.7
   - 推荐理由：三星级酒店，价格297元符合预算，评分4.7分，位于陆家嘴塘桥地铁站附近，位置优越。

3. **如家酒店(上海浦东机场妙境路川沙地铁站店)**
   - 地址：妙境路635号（浦东新区）
   - 参考价格：未显示（经济型酒店通常在预算范围内）
   - 评分：4.6
   - 推荐理由：评分4.6分，靠近川沙地铁站，经济型连锁品牌，服务标准化。

4. **如家酒店·neo(上海浦东航头地铁站店)**
   - 地址：沪南路5469弄63号（浦东新区）
   - 参考价格：未显示（经济型酒店通常在预算范围内）
   - 评分：4.6
   - 推荐理由：评分4.6分，位于航头地铁站附近，neo升级版经济型酒店，设施较新。

5. **如家酒店·neo(上海崇明八一路步行街店)**
   - 地址：城桥镇八一路338号（崇明区）
   - 参考价格：未显示（经济型酒店通常在预算范围内）
   - 评分：4.6
   - 推荐理由：评分4.6分，位于崇明区八一路步行街，适合喜欢远离市区喧嚣的旅客。

关于"安静"偏好：由于工具无法提供酒店安静程度的客观数据，以上推荐主要基于评分、价格和位置进行选择。如需更安静的住宿

{
    "name": "latc.travel-multi-agent.agent.invoke",
    "context": {
        "trace_id": "0x7ddc2857389ab2f4de296778d922b9a1",
        "span_id": "0x9915d1086f60727e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x358af03e24295870",
    "start_time": "2026-11-22T16:08:09.405853Z",
    "end_time": "2026-11-22T16:08:53.558256Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "task": {
            "task.id": "T1",
            "task.agent": "HotelAgent",
            "task.description": "\u6839\u636e\u4f4d\u7f6e\u4e0a\u6d77\u3001\u9884\u7b97\u6bcf\u665a\u4e0d\u8d85\u8fc7800\u5143\u3001\u504f\u597d\u5b89\u9759\u63a8\u8350\u9152\u5e97",
            "task.depends_on": []
        },
        "thread.id": "notebook_demo",
        "task.id": "T1",
        "task.agent": "HotelAgent",
        "task.description": "\u6839\u636e\u4f4d\u7f6e\u4e0a\u6d77\u3001\u9884\u7b97\u6bcf\u665a\u4e0d\u8d85\u8fc7800\u5143\u3001\u504f\u597d\u


[多智能体自然语言协作] T4
发送方：ItineraryAgent
接收方：Supervisor
消息内容：
基于您提供的旅行需求（2026年11月23日至25日，上海，偏好安静、舒缓节奏）以及前序任务返回的天气、酒店和餐厅数据，为您生成以下最终旅行日程与总结。

**重要提示：**
1.  **航班/交通缺失**：由于出发城市未知且未提供航班信息，本行程不包含大交通（往返上海）的具体班次建议，请您自行安排抵达上海的交通方式。
2.  **景点缺失**：前序任务中未包含具体的景点推荐数据，因此本行程侧重于“舒缓节奏”的住宿体验与餐饮探索，建议您根据天气情况在酒店周边或公园进行自由漫步。
3.  **天气预警**：行程期间气温逐日下降（从16.5°C降至11.3°C），且首尾两天有雾霾，请注意保暖及佩戴口罩。

---

### 📅 上海三日舒缓之旅日程 (2026-11-23 至 2026-11-25)

#### **第一天：2026年11月23日（星期一）**
*   **天气状况**：烟雾/霾，局部零星小雨，13.1°C - 16.5°C。
*   **出行建议**：空气质量较差且有雨，建议以室内活动或短途移动为主，携带雨具和口罩。
*   **行程安排**：
    *   **抵达与入住**：抵达上海后，建议前往酒店办理入住。考虑到“安静”偏好及避免市区喧嚣，推荐选择位于非核心商业区的酒店。
        *   *推荐住宿*：**汉庭酒店(上海青浦新城地铁站店)**（评分4.8，环境相对安静，性价比高）或 **如家酒店·neo(上海崇明八一路步行街店)**（若希望彻底远离市区喧嚣，但需注意崇明距离市区较远，交通时间成本高）。若希望交通便利且安静，**上海南泉酒店(陆家嘴塘桥地铁站店)** 也是不错选择（评分4.7，价格297元/晚）。
    *   **晚餐**：选择一家评分高且氛围舒适的餐厅。
        *   *推荐餐厅*：**海尚明粤(博园路店)**（嘉定区）。评分高达5.0，人均142.6元，适合享受高品质且安静的用餐体验。若您在浦东区域，可选择 **荣桂坊新中菜(航头店)**（评分4.4，人均153.6元）。
    *   **晚间活动**：在酒店休息，适应气温变化，避免户外长时间停留以防雾霾影响。

#### **第二天：20

=== 最终回复 ===

基于您提供的旅行需求（2026年11月23日至25日，上海，偏好安静、舒缓节奏）以及前序任务返回的天气、酒店和餐厅数据，为您生成以下最终旅行日程与总结。

**重要提示：**
1.  **航班/交通缺失**：由于出发城市未知且未提供航班信息，本行程不包含大交通（往返上海）的具体班次建议，请您自行安排抵达上海的交通方式。
2.  **景点缺失**：前序任务中未包含具体的景点推荐数据，因此本行程侧重于“舒缓节奏”的住宿体验与餐饮探索，建议您根据天气情况在酒店周边或公园进行自由漫步。
3.  **天气预警**：行程期间气温逐日下降（从16.5°C降至11.3°C），且首尾两天有雾霾，请注意保暖及佩戴口罩。

---

### 📅 上海三日舒缓之旅日程 (2026-11-23 至 2026-11-25)

#### **第一天：2026年11月23日（星期一）**
*   **天气状况**：烟雾/霾，局部零星小雨，13.1°C - 16.5°C。
*   **出行建议**：空气质量较差且有雨，建议以室内活动或短途移动为主，携带雨具和口罩。
*   **行程安排**：
    *   **抵达与入住**：抵达上海后，建议前往酒店办理入住。考虑到“安静”偏好及避免市区喧嚣，推荐选择位于非核心商业区的酒店。
        *   *推荐住宿*：**汉庭酒店(上海青浦新城地铁站店)**（评分4.8，环境相对安静，性价比高）或 **如家酒店·neo(上海崇明八一路步行街店)**（若希望彻底远离市区喧嚣，但需注意崇明距离市区较远，交通时间成本高）。若希望交通便利且安静，**上海南泉酒店(陆家嘴塘桥地铁站店)** 也是不错选择（评分4.7，价格297元/晚）。
    *   **晚餐**：选择一家评分高且氛围舒适的餐厅。
        *   *推荐餐厅*：**海尚明粤(博园路店)**（嘉定区）。评分高达5.0，人均142.6元，适合享受高品质且安静的用餐体验。若您在浦东区域，可选择 **荣桂坊新中菜(航头店)**（评分4.4，人均153.6元）。
    *   **晚间活动**：在酒店休息，适应气温变化，避免户外长时间停留以防雾霾影响。

#### **第二天：2026年11月24日（星期二）**
*   **天气状况**：多云转晴，风力较强（西北风

## 7. 停止服务并清理

结束演示后终止 A2A 子进程；各服务的 MCP 会话随进程退出自动关闭。

In [18]:
for proc in _started:
    proc.terminate()
for proc in _started:
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
_started.clear()

print("已停止 A2A 服务；各服务的 MCP 会话随进程退出自动关闭。")

已停止 A2A 服务；各服务的 MCP 会话随进程退出自动关闭。


{
    "name": "latc.travel-multi-agent.agent.invoke",
    "context": {
        "trace_id": "0x7ddc2857389ab2f4de296778d922b9a1",
        "span_id": "0xfe6a244f0a43c15d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb666ca66fdd5c316",
    "start_time": "2026-11-22T16:08:53.573609Z",
    "end_time": "2026-11-22T16:09:26.645411Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "task": {
            "task.id": "T4",
            "task.agent": "ItineraryAgent",
            "task.description": "\u7efc\u5408\u65c5\u884c\u51fa\u53d1\u65f6\u95f42026\u5e7411\u670823\u65e5\u3001\u8fd4\u56de\u65f6\u95f42026\u5e7411\u670825\u65e5\u3001\u5929\u6c14\u3001\u666f\u70b9\u3001\u9152\u5e97\u3001\u9910\u5385\u548c\u822a\u73ed\u7ed3\u679c\uff0c\u751f\u6210\u6700\u7ec8\u65c5\u884c\u65e5\u7a0b\u4e0e\u7528\u6237\u7b54\u590d",
            "task.depends_on": [
                "T1",
                "T2",
                "T3"
            ]
 

## 8. 小结

- 中心编排用 LangGraph 固定图，`route` 是独立节点并复用第 6 章 `agent_routing`。
- 子智能体先以 A2A 服务启动，中心智能体通过 Agent Card 注册、用 `A2AClient` 调用。
- MCP 工具在服务启动时绑定并保持常驻会话，stdio 进程不会每次调用都重启。
- 聚合阶段用代码选择最终回复（无聚合 LLM），职责清晰、可追溯。

## 延伸思考：中间过程文本膨胀与排障

随着接入的领域智能体数量增加、任务粒度逐渐细化，中心编排器与各智能体之间交换的中间文本会急剧增长：每一次路由都携带任务描述与参数，每一次工具调用都返回结构化结果，每一个智能体又各自产出一段总结。当这些中间产物仅以运行日志的形式顺序打印时，阅读既不直观，也难以在最终结果偏离预期时快速定位出错环节——日志是线性且冗余的，而排障需要的是按“哪一步、哪个智能体、哪一次工具调用”的维度进行切片。
